# Transition rehearsal — shared-checkpoint experiment

**Five paired seeds, one parent per seed, three continuations.** Train the original `prereq_mixed_pair_focus` curriculum to step 6,300 once; save the entire optimizer and random-number state; restore that *same state* for abrupt, gradual-bridge, and late-Binding suffixes. This notebook runs all seeds directly and skips validated completed continuations. No pilot/full modes. The existing model, dataset and optimizer settings are unchanged.

Run from `notebooks/medium_world/`. Parent checkpoints are local `.pt` files (Git-ignored); numeric results and per-example predictions are saved separately.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEEDS = [42, 43, 44, 45, 46]
BATCH_SIZE, EVAL_EVERY = 50, 100
LR, WEIGHT_DECAY = 1e-4, 0.0
MODEL_DIM, PATCH_SIZE, MODEL_DEPTH = 48, 4, 1
AUGMENT, DEVICE = True, "cuda"
MAIN_EPOCHS, COMMON_EPOCHS = 36, 9
DATA_REL = Path("outputs/medium_world/dataset")
OUT_REL = Path("outputs/medium_world/experiments/transition_rehearsal/five_seeds")
DATA_DIR, OUT_DIR = PROJECT_ROOT / DATA_REL, PROJECT_ROOT / OUT_REL
RESULTS_FILE = OUT_DIR / "schedule_comparison.json"
TRACE_DIR, CHECKPOINT_DIR = OUT_DIR / "prediction_traces", OUT_DIR / "checkpoints"

In [2]:
import copy, hashlib, json, random
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from src.medium_visual_world.learning import (
    TASKS, MediumDataset, AttributeBindingViT, augment_images, set_seed, environment
)
from src.medium_visual_world.transition_control import (
    SOURCE, STRATEGIES, PREFIX_STEPS, TOTAL_STEPS, make_transition_blocks
)
from src.medium_visual_world.prediction_tracking import (
    PredictionTrace, evaluate_predictions, load_trace
)

device = torch.device(DEVICE)
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but not available; change DEVICE to 'cpu' if needed")
torch.set_num_threads(min(4, torch.get_num_threads()))
train_data = MediumDataset(DATA_DIR / "train.jsonl")
id_dev, ood_dev = MediumDataset(DATA_DIR / "id_dev.jsonl"), MediumDataset(DATA_DIR / "ood_dev.jsonl")
datasets = {"id_dev": id_dev, "ood_dev": ood_dev}
hashes = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
          for name in ("train.jsonl", "id_dev.jsonl", "ood_dev.jsonl")}
EXPECTED_HASHES = {
    "train.jsonl": "69deccacbac6db25f7ffc8165f01545483d856855282ea640997f6660ce27c26",
    "id_dev.jsonl": "ece4b831f1f7fe6efad42b18aca499f12c2d573baa76fc218864bb56fe24b11a",
    "ood_dev.jsonl": "208a7566cc2082fa4438ec0d003430d0927c54bf82c55bb40fa885ef4961e0aa",
}
assert hashes == EXPECTED_HASHES, "Dataset differs from the completed Mixed → Pair experiment"
assert len(train_data) == 10000 and all(train_data.tasks.count(t) == 2500 for t in TASKS)
assert len(id_dev) == 1024 and len(ood_dev) == 400
print(device, "|", len(SEEDS), "seeds ×", len(STRATEGIES), "continuations | common parent step", PREFIX_STEPS)

cuda | 5 seeds × 3 continuations | common parent step 6300


## Verify frozen schedules

All suffixes share one parent, use the same per-task question streams, and preserve 45 presentations of **each exact question**. Abrupt reproduces the previous control schedule precisely.

In [3]:
def verify_real_plan(seed):
    prefix_ref = None
    for strategy in STRATEGIES:
        prefix, suffix, plan = make_transition_blocks(train_data.rows, seed, strategy,
                                                      MAIN_EPOCHS, COMMON_EPOCHS)
        assert prefix_ref is None or prefix == prefix_ref
        prefix_ref = prefix
        assert sum(len(indices) for _, indices in prefix) == PREFIX_STEPS * BATCH_SIZE
        assert sum(len(indices) for _, indices in suffix) == (TOTAL_STEPS-PREFIX_STEPS)*BATCH_SIZE
        assert all(len(indices) % BATCH_SIZE == 0 for _, indices in prefix+suffix)
        assert set(Counter(i for _, ids in prefix+suffix for i in ids).values()) == {45}
        assert all(n == 112500 for n in plan["total_presentations_by_task"].values())
    return [dict(seed=seed, strategy=s, phases=" / ".join(
        f"{r['phase']}:{r['end_step']}" for r in make_transition_blocks(train_data.rows,seed,s)[2]["phases"]))
            for s in STRATEGIES]

display(pd.DataFrame([row for seed in SEEDS for row in verify_real_plan(seed)]))
print("PASS: all schedules have matched exposure, source prefix, and exact phase boundaries")

,seed,strategy,phases
0,42,abrupt,prereq_mixed:5400 / pair_early:6300 / pair_foc...
1,42,gradual_bridge,prereq_mixed:5400 / pair_early:6300 / pair_ini...
2,42,late_binding,prereq_mixed:5400 / pair_early:6300 / pair_ini...
3,43,abrupt,prereq_mixed:5400 / pair_early:6300 / pair_foc...
4,43,gradual_bridge,prereq_mixed:5400 / pair_early:6300 / pair_ini...
5,43,late_binding,prereq_mixed:5400 / pair_early:6300 / pair_ini...
6,44,abrupt,prereq_mixed:5400 / pair_early:6300 / pair_foc...
7,44,gradual_bridge,prereq_mixed:5400 / pair_early:6300 / pair_ini...
8,44,late_binding,prereq_mixed:5400 / pair_early:6300 / pair_ini...
9,45,abrupt,prereq_mixed:5400 / pair_early:6300 / pair_foc...


PASS: all schedules have matched exposure, source prefix, and exact phase boundaries


## Save a complete parent checkpoint at step 6,300

State includes weights, AdamW moments, random-number generator states, full prefix history, and its prediction trace. Saving it once avoids training the shared history three times.

In [4]:
CONFIG = {"source": SOURCE, "strategies": list(STRATEGIES), "seeds": SEEDS,
          "main_epochs": MAIN_EPOCHS, "common_epochs": COMMON_EPOCHS,
          "batch_size": BATCH_SIZE, "eval_every": EVAL_EVERY,
          "learning_rate": LR, "weight_decay": WEIGHT_DECAY,
          "model_dim": MODEL_DIM, "patch_size": PATCH_SIZE, "depth": MODEL_DEPTH,
          "augmentation": AUGMENT, "prefix_steps": PREFIX_STEPS,
          "total_steps": TOTAL_STEPS, "trace_schema": 1}


def capture_rng():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def model_digest(model):
    digest = hashlib.sha256()
    for key, value in sorted(model.state_dict().items()):
        digest.update(key.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def new_model_optimizer():
    model = AttributeBindingViT(dim=MODEL_DIM, patch_size=PATCH_SIZE,
                                depth=MODEL_DEPTH).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    return model, optimizer


def new_trace(seed, schedule, parent_hash=None):
    return PredictionTrace(datasets, metadata={"seed": seed, "schedule": schedule,
                            "dataset_hashes": hashes, "parent_sha256": parent_hash})


def evaluate_at(model, optimizer, trace, history, step, phase, seen, train_loss=None):
    scores = {}
    for split, data in datasets.items():
        result, preds = evaluate_predictions(model, data, device, BATCH_SIZE)
        scores[split] = result
        trace.add(split, step, preds)
    history.append({"step": step, "phase": phase,
                    "lr": optimizer.param_groups[0]["lr"],
                    "batch_train_loss": train_loss,
                    "cumulative_task_presentations": dict(seen), **scores})
    print(f"{phase:16} step={step:4} ID={scores['id_dev']['by_template']['pair']:.3f} "
          f"OOD={scores['ood_dev']['by_template']['pair']:.3f}", flush=True)


def fit_blocks(model, optimizer, blocks, trace, history, start_step, seen):
    step = start_step
    transitions = []
    for phase, indices in blocks:
        phase_start = step
        accumulated, batches = 0.0, 0
        for start in range(0, len(indices), BATCH_SIZE):
            batch_ids = indices[start:start+BATCH_SIZE]
            x, q, y = train_data.batch(batch_ids, device)
            if AUGMENT:
                x = augment_images(x)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(x, q), y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at step {step}")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1
            accumulated += float(loss.item())
            batches += 1
            seen.update(train_data.tasks[i] for i in batch_ids)
            if step % EVAL_EVERY == 0:
                evaluate_at(model, optimizer, trace, history, step, phase, seen,
                            accumulated/batches)
                accumulated, batches = 0.0, 0
        assert history[-1]["step"] == step
        transitions.append({"phase":phase,"start_step":phase_start,"end_step":step})
    return step, transitions


def prefix_path(seed):
    return CHECKPOINT_DIR / f"prefix_seed_{seed}.pt"


def prefix_trace_path(seed):
    return TRACE_DIR / f"prefix_seed_{seed}.npz"


def load_parent(seed):
    path = prefix_path(seed)
    if not path.exists() or not prefix_trace_path(seed).exists():
        return None
    # Local checkpoint created by this notebook; never load untrusted .pt files.
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    assert ckpt["seed"] == seed and ckpt["step"] == PREFIX_STEPS
    assert ckpt["config"] == CONFIG and ckpt["dataset_hashes"] == hashes
    t = load_trace(prefix_trace_path(seed))
    for split,data in datasets.items():
        block=t["splits"][split]
        assert np.array_equal(block["steps"],np.arange(0,PREFIX_STEPS+1,EVAL_EVERY))
        assert list(block["ids"]) == [str(r["id"]) for r in data.rows]
    assert len(ckpt["history"]) == PREFIX_STEPS//EVAL_EVERY+1
    return ckpt


def make_parent(seed):
    found = load_parent(seed)
    if found is not None:
        print(f"Reusing verified parent checkpoint: seed {seed}")
        return found
    set_seed(seed)
    model, optimizer = new_model_optimizer()
    prefix,_,_ = make_transition_blocks(train_data.rows, seed, STRATEGIES[0])
    trace = new_trace(seed, "prefix")
    history, seen = [], Counter()
    evaluate_at(model, optimizer, trace, history, 0, "initial", seen)
    step, transitions = fit_blocks(model, optimizer, prefix, trace, history, 0, seen)
    assert step == PREFIX_STEPS
    parent_hash = model_digest(model)
    ckpt = {"seed": seed, "step": step, "config": CONFIG,
            "dataset_hashes": hashes, "model": model.state_dict(),
            "optimizer": optimizer.state_dict(), "rng": capture_rng(),
            "parent_sha256": parent_hash, "history": history,
            "seen": dict(seen), "transitions": transitions}
    TRACE_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    trace.save(prefix_trace_path(seed))
    tmp = prefix_path(seed).with_suffix(".pt.tmp")
    torch.save(ckpt, tmp)
    tmp.replace(prefix_path(seed))
    print(f"Saved shared checkpoint: seed {seed} | SHA {parent_hash[:12]}")
    return ckpt

## Branch into three independent continuations

Each branch restores exactly the same parent state, *including* the optimizer and augmentation RNG. Results are written after each finished seed–strategy run; reruns verify the saved prediction traces before skipping.

In [5]:
def trace_from_parent(seed, strategy, parent_hash):
    prefix = load_trace(prefix_trace_path(seed))
    trace = new_trace(seed, strategy, parent_hash)
    for split in datasets:
        block = prefix["splits"][split]
        for index, step in enumerate(block["steps"]):
            trace.add(split, int(step), {"predicted_label": block["predictions"][index],
                                         "probability_yes": block["probabilities"][index]})
    return trace


def trace_path(seed, strategy):
    return TRACE_DIR / f"{strategy}_seed_{seed}.npz"


def verify_trace(seed, strategy, parent_hash):
    trace = load_trace(trace_path(seed, strategy))
    assert trace["metadata"]["seed"] == seed and trace["metadata"]["schedule"] == strategy
    assert trace["metadata"]["parent_sha256"] == parent_hash
    assert trace["metadata"]["dataset_hashes"] == hashes
    for split, data in datasets.items():
        block = trace["splits"][split]
        assert list(block["ids"]) == [str(row["id"]) for row in data.rows]
        assert np.array_equal(block["steps"], np.arange(0,TOTAL_STEPS+1,EVAL_EVERY))
        assert block["predictions"].shape == (91,len(data))
        assert np.isfinite(block["probabilities"]).all()
    return trace


def binding_auc(history, begin=0, end=TOTAL_STEPS, split="id_dev"):
    subset = [row for row in history if begin <= row["step"] <= end]
    assert subset[0]["step"] == begin and subset[-1]["step"] == end
    return sum((b["step"]-a["step"])*(a[split]["by_template"]["pair"]+b[split]["by_template"]["pair"])/2
               for a,b in zip(subset[:-1],subset[1:]))/(end-begin)


def continue_one(seed, strategy, parent):
    assert parent["step"] == PREFIX_STEPS
    prefix, suffix, plan = make_transition_blocks(train_data.rows, seed, strategy)
    # Build model/optimizer *before* restoring saved RNG states.
    model, optimizer = new_model_optimizer()
    model.load_state_dict(parent["model"])
    optimizer.load_state_dict(copy.deepcopy(parent["optimizer"]))
    assert model_digest(model) == parent["parent_sha256"]
    restore_rng(parent["rng"])
    trace = trace_from_parent(seed, strategy, parent["parent_sha256"])
    history, seen = copy.deepcopy(parent["history"]), Counter(parent["seen"])
    step, transitions = fit_blocks(model, optimizer, suffix, trace, history,
                                   PREFIX_STEPS, seen)
    assert step == TOTAL_STEPS and len(history) == 91
    assert all(seen[task] == 112500 for task in TASKS)
    trace.save(trace_path(seed, strategy))
    verify_trace(seed, strategy, parent["parent_sha256"])
    return {"seed": seed, "schedule": strategy, "status":"completed",
            "steps_run": step, "parent_seed": seed,
            "parent_sha256": parent["parent_sha256"],
            "phase_plan": plan,
            "transitions": parent["transitions"] + transitions,
            "history": history, "final": history[-1],
            "id_pair_aulc": binding_auc(history),
            "id_pair_suffix_aulc": binding_auc(history, PREFIX_STEPS),
            "ood_pair_suffix_aulc": binding_auc(history, PREFIX_STEPS, split="ood_dev"),
            "prediction_trace": trace_path(seed,strategy).relative_to(OUT_DIR).as_posix()}


def write_report(report):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    temp = RESULTS_FILE.with_suffix(".json.tmp")
    temp.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
    temp.replace(RESULTS_FILE)

if RESULTS_FILE.exists():
    report = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))
    if report["config"] != CONFIG or report["dataset_hashes"] != hashes:
        raise ValueError("Existing results use different configuration/dataset hashes")
else:
    report = {"status": "exploratory", "config": CONFIG, "dataset_hashes": hashes,
              "environment": environment(), "runs": []}

existing = {(r["seed"],r["schedule"]):r for r in report["runs"]}
assert len(existing) == len(report["runs"]), "Duplicate completed result entries"
for seed in SEEDS:
    parent = make_parent(seed)
    for strategy in STRATEGIES:
        key = (seed,strategy)
        if key in existing:
            assert existing[key]["status"] == "completed"
            assert existing[key]["parent_sha256"] == parent["parent_sha256"]
            verify_trace(seed,strategy,parent["parent_sha256"])
            print("Skipping verified run:",seed,strategy)
            continue
        result = continue_one(seed,strategy,parent)
        report["runs"].append(result)
        existing[key] = result
        write_report(report)
        print("Saved",seed,strategy,"→",RESULTS_FILE)
assert len(existing) == len(SEEDS)*len(STRATEGIES)
print("PASS:",len(existing),"completed runs, 5 shared parent checkpoints")

initial          step=   0 ID=0.500 OOD=0.500
prereq_mixed     step= 100 ID=0.492 OOD=0.500
prereq_mixed     step= 200 ID=0.508 OOD=0.500
prereq_mixed     step= 300 ID=0.512 OOD=0.500
prereq_mixed     step= 400 ID=0.508 OOD=0.480
prereq_mixed     step= 500 ID=0.508 OOD=0.482
prereq_mixed     step= 600 ID=0.504 OOD=0.487
prereq_mixed     step= 700 ID=0.504 OOD=0.485
prereq_mixed     step= 800 ID=0.500 OOD=0.482
prereq_mixed     step= 900 ID=0.500 OOD=0.482
prereq_mixed     step=1000 ID=0.500 OOD=0.482
prereq_mixed     step=1100 ID=0.496 OOD=0.497
prereq_mixed     step=1200 ID=0.508 OOD=0.495
prereq_mixed     step=1300 ID=0.500 OOD=0.492
prereq_mixed     step=1400 ID=0.504 OOD=0.492
prereq_mixed     step=1500 ID=0.512 OOD=0.495
prereq_mixed     step=1600 ID=0.508 OOD=0.495
prereq_mixed     step=1700 ID=0.508 OOD=0.495
prereq_mixed     step=1800 ID=0.508 OOD=0.492
prereq_mixed     step=1900 ID=0.504 OOD=0.492
prereq_mixed     step=2000 ID=0.504 OOD=0.492
prereq_mixed     step=2100 ID=0.50

## Verify final consistency

The parent model SHA should be **identical across the three branches of each seed**. All branches finish at step 9,000 with complete 91-checkpoint prediction traces. Analyze with `transition_rehearsal_analysis.ipynb`.

In [6]:
rows=[]
for r in report["runs"]:
    trace=verify_trace(r["seed"],r["schedule"],r["parent_sha256"])
    rows.append({"seed":r["seed"],"strategy":r["schedule"],
                 "parent_sha":r["parent_sha256"][:12],
                 "id_binding":r["final"]["id_dev"]["by_template"]["pair"],
                 "ood_binding":r["final"]["ood_dev"]["by_template"]["pair"],
                 "trace_checkpoints":len(trace["splits"]["id_dev"]["steps"])})
summary=pd.DataFrame(rows)
assert summary.groupby("seed")["parent_sha"].nunique().eq(1).all()
display(summary)
print("PASS: all branch groups share their exact starting weights and prediction traces are complete")

,seed,strategy,parent_sha,id_binding,ood_binding,trace_checkpoints
0,42,abrupt,d5608a38fc99,0.648438,0.6075,91
1,42,gradual_bridge,d5608a38fc99,0.687500,0.6350,91
2,42,late_binding,d5608a38fc99,0.691406,0.6300,91
3,43,abrupt,1bc72a828744,0.597656,0.5325,91
4,43,gradual_bridge,1bc72a828744,0.632812,0.5775,91
5,43,late_binding,1bc72a828744,0.656250,0.6100,91
6,44,abrupt,b1b91b420b42,0.625000,0.5725,91
7,44,gradual_bridge,b1b91b420b42,0.628906,0.5650,91
8,44,late_binding,b1b91b420b42,0.636719,0.5825,91
9,45,abrupt,36c41e755797,0.621094,0.5675,91


PASS: all branch groups share their exact starting weights and prediction traces are complete
